<a href="https://colab.research.google.com/github/heyitsarun/Aero-Gen-Synthetic-Anomaly-Generation-in-Aviation-Communications/blob/main/02_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/aerogen_project'
DATA_DIR = f'{PROJECT_ROOT}/data'
GEN_DIR = f'{PROJECT_ROOT}/generated'
os.makedirs(GEN_DIR, exist_ok=True)

df = pd.read_csv(f'{DATA_DIR}/atcosim_enriched.csv')
asrs_df = pd.read_csv(f'{DATA_DIR}/asrs_clean_taxonomy.csv')
print(f'Loaded {len(df)} enriched ATCOSIM rows, {len(asrs_df)} ASRS taxonomy rows')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 6519 enriched ATCOSIM rows, 45 ASRS taxonomy rows


In [2]:
NUMBER_WORDS_LIST = ['zero','one','two','three','four','five','six','seven','eight','nine']
TRAILING_STRIP_WORDS = {'radar', 'control', 'approach', 'departure', 'tower', 'ground', 'in', 'is'}
LEADING_STRIP_WORDS = {'break'}

def clean_callsign(callsign):
    tokens = callsign.split()
    while tokens and tokens[0] in LEADING_STRIP_WORDS:
        tokens = tokens[1:]
    while tokens and tokens[-1] in TRAILING_STRIP_WORDS:
        tokens = tokens[:-1]
    return ' '.join(tokens)

def extract_airline_and_number(callsign):
    tokens = callsign.split()
    airline_tokens, number_tokens = [], []
    seen_number = False
    for tok in tokens:
        if tok in NUMBER_WORDS_LIST:
            seen_number = True
            number_tokens.append(tok)
        elif not seen_number:
            airline_tokens.append(tok)
        else:
            number_tokens.append(tok)
    return ' '.join(airline_tokens), number_tokens

df['callsign'] = df['callsign'].apply(clean_callsign)
df[['airline', 'flight_number_tokens']] = df['callsign'].apply(
    lambda c: pd.Series(extract_airline_and_number(c))
)

CONFIRMED_AIRLINE_THRESHOLD = 5
airline_frequency = df['airline'].value_counts()
confirmed_airlines = set(airline_frequency[airline_frequency >= CONFIRMED_AIRLINE_THRESHOLD].index)

valid_swap_pool = df[
    df['airline'].isin(confirmed_airlines) &
    df['flight_number_tokens'].apply(lambda toks: any(t in NUMBER_WORDS_LIST for t in toks))
].reset_index(drop=True)

print(f'{len(confirmed_airlines)} confirmed airlines, {len(valid_swap_pool)} rows in final swap pool')

134 confirmed airlines, 5679 rows in final swap pool


In [3]:
def corrupt_same_company_similar_number(row, swap_pool):
    airline = row['airline']
    same_airline_rows = swap_pool[(swap_pool['airline'] == airline) & (swap_pool['callsign'] != row['callsign'])]
    if len(same_airline_rows) == 0:
        return None
    swap_target = same_airline_rows.sample(1).iloc[0]
    corrupted_callsign = swap_target['callsign']
    corrupted_text = row['text'].replace(row['callsign'], corrupted_callsign, 1)
    return {'anomaly_type': 'callsign_confusion_same_company', 'original_text': row['text'],
            'corrupted_text': corrupted_text, 'original_callsign': row['callsign'],
            'corrupted_callsign': corrupted_callsign, 'source_pattern': 'ASRS: same-company similar flight number'}

def corrupt_similar_sounding_different_operator(row, swap_pool):
    other_airlines = swap_pool[swap_pool['airline'] != row['airline']]['airline'].unique()
    if len(other_airlines) == 0:
        return None
    target_airline = random.choice(other_airlines)
    swap_target = swap_pool[swap_pool['airline'] == target_airline].sample(1).iloc[0]
    corrupted_callsign = swap_target['callsign']
    corrupted_text = row['text'].replace(row['callsign'], corrupted_callsign, 1)
    return {'anomaly_type': 'callsign_confusion_different_operator', 'original_text': row['text'],
            'corrupted_text': corrupted_text, 'original_callsign': row['callsign'],
            'corrupted_callsign': corrupted_callsign, 'source_pattern': 'ASRS: similar-sounding callsign, different operator'}

def corrupt_no_callsign_used(row):
    if not row['instruction']:
        return None
    return {'anomaly_type': 'no_callsign_used', 'original_text': row['text'],
            'corrupted_text': row['instruction'], 'original_callsign': row['callsign'],
            'corrupted_callsign': '', 'source_pattern': 'ASRS: instruction issued without stating callsign'}

def corrupt_readback_digit_error(row):
    tokens = row['instruction'].split()
    number_positions = [i for i, t in enumerate(tokens) if t in NUMBER_WORDS_LIST]
    if not number_positions:
        return None
    pos = random.choice(number_positions)
    new_digit = random.choice([d for d in NUMBER_WORDS_LIST if d != tokens[pos]])
    corrupted_tokens = tokens.copy()
    corrupted_tokens[pos] = new_digit
    corrupted_instruction = ' '.join(corrupted_tokens)
    corrupted_text = row['text'].replace(row['instruction'], corrupted_instruction, 1)
    return {'anomaly_type': f"readback_error_{row['value_type']}", 'original_text': row['text'],
            'corrupted_text': corrupted_text, 'original_callsign': row['callsign'],
            'corrupted_callsign': row['callsign'], 'source_pattern': f"ASRS: digit misread in {row['value_type']} readback"}

In [4]:
RULE_BASED_PATH = f'{GEN_DIR}/synthetic_rule_based.csv'
synthetic_df = pd.read_csv(RULE_BASED_PATH)
print(f'Loaded existing rule-based set: {len(synthetic_df)} rows')
print(synthetic_df['anomaly_type'].value_counts())

Loaded existing rule-based set: 1312 rows
anomaly_type
no_callsign_used                         286
callsign_confusion_different_operator    272
readback_error_altitude                  232
readback_error_frequency                 229
callsign_confusion_same_company          202
readback_error_heading                    46
readback_error_squawk_code                27
readback_error_unclassified               18
Name: count, dtype: int64


In [5]:
LLM_OUTPUT_PATH = f'{GEN_DIR}/synthetic_llm_generated.csv'
llm_synthetic_df = pd.read_csv(LLM_OUTPUT_PATH)
before = len(llm_synthetic_df)

llm_synthetic_df = llm_synthetic_df[
    llm_synthetic_df['anomaly_type'] != 'readback_error_altitude'
].reset_index(drop=True)

print(f'Removed {before - len(llm_synthetic_df)} readback_error_altitude rows (pre-fix, unreliable)')
llm_synthetic_df.to_csv(LLM_OUTPUT_PATH, index=False)
print(llm_synthetic_df['anomaly_type'].value_counts())

Removed 60 readback_error_altitude rows (pre-fix, unreliable)
anomaly_type
callsign_confusion_same_company          60
callsign_confusion_different_operator    60
no_callsign_used                         60
missing_similarity_warning               60
Name: count, dtype: int64


## Part B — LLM Few-Shot Generator
Uses the ASRS taxonomy narratives as grounding examples to generate anomalies in ATCOSIM-style phraseology, covering the same anomaly types as the rule-based baseline plus the "missing similarity warning" pattern.

In [19]:
!pip install -q groq

from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.2 MB/s eta 0:00:00
Hello, it's nice to meet you and I'm looking forward to our conversation.


Using Groq as it is the only LLM that offers free tier.